# Ideas planner, second list, idea 7: LoFTR against large map windows for skipped photos
Today a candidate needs the coarse stage to name a tile. Here only the right ~1.1 km BLOCK has to be among the best few: rank the blocks of the coarse map by their best score, cut a 640 px window (at half resolution, so about 1.1 km at level 3) around the block centre, and let LoFTR place the half-resolution photo inside it.
Reading, per photo: (1) the rank of the block that holds the truth among all blocks; (2) the best RANSAC inliers of LoFTR on the truth block window against the inliers on the windows of the 5 best OTHER blocks.
Photos: NEVER (SEED_IDS_FILE) plus controls retrieved today. Falsification (planned): dead if the truth block is outside the best 5 for most photos, or if the inliers on the truth window stay below 8.

In [1]:
import os, sys, time, gc, functools
from pathlib import Path
import numpy as np
import pandas as pd
import torch

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)
T7 = ROOT / "tests/Test_Iteration_7/results"

DATA = os.environ.get("GATE_DATA", "official")          # official or nudt
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))
TAG = os.environ.get("TAG", "official150")              # which rich dump (results/rich_dump/<TAG>) names the photos
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))           # >0: only the first photos (smoke test)
print(DATA, TAG, "pool", POOL_PHOTOS, "limit", N_LIMIT)

official official150 pool 500 limit 0


In [2]:
if DATA == "nudt":
    from train.datasets import photo_sets
    photo_sets.load_nudt_photos = functools.partial(photo_sets.load_nudt_photos, split=os.environ.get("NUDT_SPLIT", "test"))
from train.checks.rich_dump import RichDumpPipeline
from train.checks import replay
from coarse_search.heading_queries import query_geometry
from image_processing.geometry import render_query
from image_processing.map_levels import level_for_height, level_gsd

def make_pipeline(headings, scales, confidence=0.22, coarse_threshold=0.21):
    pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=1.0, k_peaks=50, m_fine=10,
                                matcher="loftr_minima", min_inliers=8, ransac_pixels=None, matcher_confidence=confidence, satellite="new",
                                map_margin_m=None, loftr_coarse_threshold=coarse_threshold, dino_weight=None, headings=headings, scale_factors=scales)
    start = time.time()
    pipeline.setup(POOL_PHOTOS, 7)
    print("setup %.0f s" % (time.time() - start))
    return pipeline

STANDARD = [(-5.0, 1.0), (0.0, 1.0), (5.0, 1.0), (-5.0, 1.05), (0.0, 1.05), (5.0, 1.05)]
dump = replay.load_set(T7 / "rich_dump" / TAG)

In [3]:
pipeline = make_pipeline([-5.0, 0.0, 5.0], [1.0, 1.05])
from train.checks import part_maps
from fine_matching.ransac import fit_transform
from fine_matching.loftr_matcher import loftr_matches
import cv2
never = [int(i) for i in pd.read_csv(os.environ["SEED_IDS_FILE"]).photo]
control = [p for p in dump if p not in never][: int(os.environ.get("CONTROL_N", "10"))]
if N_LIMIT:
    never, control = never[:N_LIMIT], control[:N_LIMIT]
test_set = pipeline.test_set
BLOCK_M, WINDOW = 1100.0, 640
print(len(never), "never retrieved;", len(control), "controls")

level 3: 0.85 m/pixel, picture 16378 x 16005, map coverage 100%
level 4: 1.20 m/pixel, picture 11581 x 11317, map coverage 100%
level 5: 1.70 m/pixel, picture 8189 x 8002, map coverage 100%
level 6: 2.40 m/pixel, picture 5790 x 5658, map coverage 100%
level 7: 3.39 m/pixel, picture 4094 x 4001, map coverage 100%


map store ready, took 210 s


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup 417 s
12 never retrieved; 8 controls


In [4]:
def map_window(level_images, level, x_m, y_m, size):
    """A size x size piece of the level image centred on (x_m, y_m); zero where outside the map."""
    from fine_matching.loftr_matcher import map_piece
    return map_piece(level_images, level, x_m, y_m, size)

def block_ranks(score_map, geometry, level, truth):
    """Block maximum of the score map over square blocks of BLOCK_M metres; returns (ranks of the blocks by score, truth block id, block centres)."""
    cell_m = float(geometry["x_m"][1] - geometry["x_m"][0])
    block = max(1, int(round(BLOCK_M / cell_m)))
    usable = score_map.clone(); usable[~geometry["inside"]] = -1.0
    pooled = torch.nn.functional.max_pool2d(usable[None, None], block, stride=block, ceil_mode=True)[0, 0]
    flat_order = torch.argsort(pooled.reshape(-1), descending=True).cpu().numpy()
    cols = pooled.shape[1]
    centre_x = lambda c: float(geometry["x_m"][min(c * block + block // 2, len(geometry["x_m"]) - 1)])
    centre_y = lambda r: float(geometry["y_m"][min(r * block + block // 2, len(geometry["y_m"]) - 1)])
    x0 = float(geometry["x_m"][0]); y0 = float(geometry["y_m"][0])
    truth_col = int((truth[0] - x0) // (block * cell_m)); truth_row = int((truth[1] - y0) // (block * cell_m))
    truth_id = truth_row * cols + truth_col
    order = [(int(i) // cols, int(i) % cols) for i in flat_order if float(pooled.reshape(-1)[int(i)]) > -1]
    rank = [i for i, f in enumerate(flat_order) if int(f) == truth_id]
    return order, (truth_row, truth_col), (centre_x, centre_y), (int(rank[0]) + 1 if rank else 0)

def window_inliers(canvas_half, level, x_m, y_m):
    window = map_window(pipeline.level_images, level, x_m, y_m, 2 * WINDOW)
    window = cv2.resize(window, (WINDOW, WINDOW), interpolation=cv2.INTER_AREA)
    big = np.zeros((WINDOW, WINDOW), np.float32)                        # LoFTR needs two equal square pictures: the half-resolution photo sits in the middle of a black one
    height, width = canvas_half.shape
    top, left = (WINDOW - height) // 2, (WINDOW - width) // 2
    big[top:top + height, left:left + width] = canvas_half
    pa, pb, conf = loftr_matches(pipeline.matcher, big, window.astype(np.float32))
    keep = conf >= 0.22
    if keep.sum() < 4:
        return 0
    matrix, inliers = fit_transform(pa[keep], pb[keep], None)
    return int(inliers.sum()) if matrix is not None else 0

rows = []
start = time.time()
for group, ids in [("never", never), ("control", control)]:
    for pid in ids:
        item = test_set.items[pid]; prior = test_set.priors[pid]; truth = [float(v) for v in test_set.truths[pid]]
        level = level_for_height(item["rel_height"], pipeline.camera)
        for h, s in STANDARD[:3]:
            canvas, valid = render_query(item, pipeline.camera, level, query_geometry(h, s))
            score_map, geometry = part_maps.zncc_map(pipeline.store, canvas, valid, level, prior)
            order, truth_block, centres, rank = block_ranks(score_map, geometry, level, truth)
            half = cv2.resize(canvas, (canvas.shape[1] // 2, canvas.shape[0] // 2), interpolation=cv2.INTER_AREA)
            truth_inl = window_inliers(half, level, truth[0], truth[1])      # window centred on the truth (best case of the truth block)
            false_inl = []
            for (r, c) in [b for b in order if b != truth_block][:5]:
                false_inl.append(window_inliers(half, level, centres[0](c), centres[1](r)))
            rows.append(dict(group=group, photo=pid, height=item["rel_height"], heading=h, scale=s, truth_block_rank=rank, blocks=len(order),
                             truth_window_inliers=truth_inl, false_window_max=max(false_inl) if false_inl else 0))
            del score_map; torch.cuda.empty_cache(); gc.collect()
        print(group, pid, "%.0f s" % (time.time() - start), flush=True)
W = pd.DataFrame(rows)
W.to_csv(T7 / ("idea7b_large_window_%s.csv" % TAG), index=False)
P = W.groupby(["group", "photo"]).agg(best_truth_block_rank=("truth_block_rank", lambda r: int(r[r > 0].min()) if (r > 0).any() else 0),
                                      truth_window_inliers=("truth_window_inliers", "max"), false_window_max=("false_window_max", "max")).reset_index()
P.to_csv(T7 / ("idea7b_large_window_photos_%s.csv" % TAG), index=False)
print(P.to_string(index=False))

never 11 7 s


never 35 12 s


never 66 18 s


never 172 24 s


never 176 30 s


never 200 35 s


... (11 outputs left out)

control 7 108 s


control 8 114 s


control 9 119 s


  group  photo  best_truth_block_rank  truth_window_inliers  false_window_max
control      1                      1                     0                 0
control      2                      1                     5                 0
control      4                      1                     0                 0
control      5                      1                    26                11
control      6                      1                     0                 0
control      7                      1                     0                 0
control      8                      1                     0                 0
control      9                      1                     0                 0
  never     11                     19                     0                 0
  never     35                      1                     0                 0
  never     66                      1                     0                 0
  never    172                      4                     0     